# 1. Cleaning

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
df = pd.read_csv('loan_logreg.csv', encoding = 'utf-8')

In [3]:
pd.set_option('display.max_columns', None)
df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303638 entries, 0 to 1303637
Data columns (total 26 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   loan_amnt             1303638 non-null  int64  
 1   is_term_sixty         1303638 non-null  int64  
 2   purpose               1303638 non-null  object 
 3   is_whole_offer        1303638 non-null  int64  
 4   is_joint_application  1303638 non-null  int64  
 5   emp_length            1228181 non-null  float64
 6   home_ownership        1303638 non-null  object 
 7   annual_inc            1303638 non-null  int64  
 8   verification_status   1303638 non-null  object 
 9   verif_status          1303638 non-null  int64  
 10  addr_state            1303638 non-null  object 
 11  delinq_2yrs           1303638 non-null  int64  
 12  inq_last_6mths        1303638 non-null  int64  
 13  pub_rec               1303638 non-null  int64  
 14  pub_rec_bankruptcies  1303638 non-

In [4]:
df = df.drop(columns=['verification_status'])

In [5]:
s = df["emp_length"].value_counts(normalize=True)
missing = df['emp_length'].isnull()
df.loc[missing,'emp_length'] = np.random.choice(s.index, size=len(df[missing]),p=s.values)

In [6]:
df.head(20)

,loan_amnt,is_term_sixty,purpose,is_whole_offer,is_joint_application,emp_length,home_ownership,annual_inc,verif_status,addr_state,delinq_2yrs,inq_last_6mths,pub_rec,pub_rec_bankruptcies,total_acc,mort_acc,dti,open_acc,revol_bal,revol_util,acc_now_delinq,tot_coll_amt,tot_cur_bal,is_default,credit_hist_months
0,30000,0,debt_consolidation,1,1,5.0,MORTGAGE,100000,1,CA,0,0,1,1,19,1,30.46,11,15603,37.0,0,0,472330,0,83
1,40000,1,credit_card,1,1,0.0,MORTGAGE,45000,2,OH,0,0,0,0,37,1,50.53,18,34971,64.5,0,0,271068,0,114
2,20000,0,credit_card,1,1,10.0,MORTGAGE,100000,0,WA,0,0,0,0,19,5,18.92,9,25416,29.9,0,0,515779,0,238
3,4500,0,credit_card,1,0,10.0,RENT,38500,0,TX,0,0,0,0,25,0,4.64,12,4472,15.3,0,306,29137,0,180
4,8425,0,credit_card,1,1,3.0,MORTGAGE,450000,2,MA,0,0,0,0,37,4,12.37,21,36812,65.7,0,0,690320,0,254
5,20000,1,debt_consolidation,1,0,4.0,RENT,57000,0,WA,0,0,0,0,16,1,22.18,10,33356,45.9,0,0,33356,0,284
6,6600,0,credit_card,0,0,10.0,RENT,45000,0,TX,0,0,0,0,13,0,16.21,6,6167,58.2,0,0,26836,0,116
7,2500,0,other,0,0,5.0,RENT,42000,0,PA,0,0,0,0,6,0,15.09,3,3319,31.6,0,0,18649,0,177
8,4000,0,house,1,0,5.0,MORTGAGE,60000,1,TX,0,1,0,0,18,2,19.10,11,382,7.3,0,0,106556,0,138
9,2700,0,credit_card,0,0,4.0,OWN,24000,0,MI,0,0,1,1,28,0,24.95,21,2696,36.4,0,449,75363,0,194


In [7]:
from scipy.stats import mannwhitneyu, chi2_contingency

isdef = df[df["is_default"] == 1]
non = df[df["is_default"] == 0]

In [8]:
print(len(isdef), len(non))

261686 1041952


In [9]:
num_features = ["loan_amnt", "emp_length", "annual_inc", "verif_status", "delinq_2yrs", "inq_last_6mths", "pub_rec", "pub_rec_bankruptcies", "total_acc", "mort_acc", "dti", "open_acc", "revol_bal", "revol_util", "acc_now_delinq", "tot_coll_amt", "tot_cur_bal", "credit_hist_months"]

bin_features = ["is_term_sixty", "is_whole_offer", "is_joint_application"]

cat_features = ["purpose", "home_ownership", "addr_state"]

In [12]:
def rank_biserial(g1, g2, u_stat):
    # effect size for mann-whitney, ranges -1 to 1
    return 1 - (2 * u_stat) / (len(g1) * len(g2))


def cramers_v(chi2_stat, n, shape):
    min_dim = min(shape[0] - 1, shape[1] - 1)
    return np.sqrt(chi2_stat / (n * min_dim))


def run_mwu(features):
    out = []
    for f in features:
        if f not in df.columns:
            continue
        u, p = mannwhitneyu(isdef[f], non[f], alternative="two-sided")
        out.append({
            "feature":   f,
            "isdef_median": round(isdef[f].median(), 4),
            "non_median": round(non[f].median(), 4),
            "U":         round(u, 1),
            "rank_biserial": round(rank_biserial(isdef[f], non[f], u), 4),
            "p_value":   p,
        })
    return pd.DataFrame(out)


def run_chi2_bin(features):
    n_total = len(df)
    out = []
    for f in features:
        if f not in df.columns:
            continue
        table = pd.crosstab(df["is_default"], df[f])
        chi2, p, dof, _ = chi2_contingency(table)
        out.append({
            "feature":     f,
            "isdef_rate":    round(isdef[f].mean(), 4),   # proportion flagged in isdef
            "non_rate":    round(non[f].mean(), 4),
            "chi2":        round(chi2, 2),
            "cramers_v":  round(cramers_v(chi2, n_total, table.shape), 4),
            "p_value":     f'{p:.3e}',
        })
    return pd.DataFrame(out)

def run_chi2_cat(features):
    n_total = len(df)
    out = []
    for f in features:
        if f not in df.columns:
            continue
        table = pd.crosstab(df["is_default"], df[f])
        chi2, p, dof, _ = chi2_contingency(table)
        out.append({
            "feature":    f,
            "chi2":       round(chi2, 2),
            "cramers_v":  round(cramers_v(chi2, n_total, table.shape), 4),
            "p_value":    f'{p:.3e}',
            "n_levels":   table.shape[1],
        })
    return pd.DataFrame(out)

In [16]:
pd.reset_option('float_format')
print("=== INDEPENDENT CONTINUOUS FEATURES (Mann-Whitney U) ===")
cont_results = run_mwu(num_features)
cont_sorted = cont_results.sort_values("p_value")
print(cont_sorted.assign(p_value=cont_sorted["p_value"].map(lambda p: f"{p:.3e}"))
      .to_string(index=False), "\n")

print("=== INDEPENDENT BINARY FEATURES (Chi-squared) ===")
bin_results = run_chi2_bin(bin_features)
print(bin_results.sort_values("p_value").to_string(index=False), "\n")

print("=== INDEPENDENT CATEGORICAL FEATURES (Chi-squared) ===")
cat_results = run_chi2_cat(cat_features)
print(bin_results.sort_values("p_value").to_string(index=False), "\n")

=== INDEPENDENT CONTINUOUS FEATURES (Mann-Whitney U) ===
             feature  isdef_median  non_median            U  rank_biserial    p_value
           loan_amnt      14350.00    12000.00 1.501466e+11        -0.1013  0.000e+00
          revol_util         55.70       51.40 1.481982e+11        -0.0870  0.000e+00
                 dti         19.77       17.09 1.576540e+11        -0.1564  0.000e+00
            mort_acc          1.00        1.00 1.225827e+11         0.1009  0.000e+00
         tot_cur_bal      56789.50    75228.50 1.268823e+11         0.0693  0.000e+00
  credit_hist_months        171.00      179.00 1.286750e+11         0.0562  0.000e+00
        verif_status          1.00        1.00 1.529311e+11        -0.1218  0.000e+00
          annual_inc      60000.00    65000.00 1.230506e+11         0.0974  0.000e+00
      inq_last_6mths          0.00        0.00 1.476539e+11        -0.0830  0.000e+00
             pub_rec          0.00        0.00 1.403898e+11        -0.0298 1.074e-2

In [ ]:
# bonferroni correction
n_tests = len(cont_results) + len(bin_results)
alpha_adj = 0.05 / n_tests
print(f"tests run: {n_tests}")
print(f"bonferroni-adjusted alpha: {round(alpha_adj, 5)}")
print("(features below this p-value are significant after correction)")